# Notebook 03 — RL Expansion: Reward-Weighted Sampling

**Phase 5** of the Protein Binder Evaluation & RL-Guided Design Pipeline.

This notebook demonstrates two RL approaches for optimising the sampling hyperparameters to maximise in silico hit rate:

1. **REINFORCE**: Reward-weighted gradient update of ProteinMPNN temperature.
2. **Bayesian Optimisation**: GP-based search over (temperature, diffusion_steps, noise_scale).

Both use the Phase 3 evaluation pipeline as the reward signal — in silico iPTM score and hit rate.

---

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
from pathlib import Path

from src.target_prep import TargetProtein, BENCHMARK_TARGETS
from src.generate import BinderGenerator
from src.evaluate import BinderEvaluator
from src.rl_sampler import ReinforceRLSampler, BayesianOptimiser, run_rl_pipeline

figures_dir = Path('../figures/rl')
figures_dir.mkdir(parents=True, exist_ok=True)

print('RL module loaded.')

In [ ]:
# Set up target, generator, evaluator
TARGET_NAME = 'EGFR'
meta = BENCHMARK_TARGETS[TARGET_NAME]

target = TargetProtein(pdb_id=meta['pdb_id'], chain_id=meta['chain_id'])
target.download_pdb()
target.parse_structure()
target.identify_hotspots()

generator = BinderGenerator(
    target=target,
    output_dir=f'../data/generated/{TARGET_NAME}_rl',
)

evaluator = BinderEvaluator(
    target=target,
    backend='mock',  # Change to 'chai1' or 'esmfold' with GPU
    thresholds={'iptm_min': 0.55, 'plddt_min': 70.0, 'interface_rmsd_max': 2.0, 'pae_max': 10.0},
)

print(f'Setup complete: {target}')

## 5.1 Baseline Evaluation (Pre-RL)

Generate a baseline batch at the default temperature T=0.1 to establish the starting hit rate.

In [ ]:
BATCH_SIZE = 32  # Increase to 100+ for real experiments
N_SEEDS = 3       # Reduce for speed; use 5 for final results

print('=== BASELINE (T=0.1) ===')

baseline_backbones = generator.run_rfdiffusion(
    num_designs=max(BATCH_SIZE // 4, 4),
    diffusion_steps=50,
    binder_length=70,
)

baseline_candidates = generator.run_proteinmpnn(
    backbone_pdbs=baseline_backbones,
    seqs_per_backbone=max(BATCH_SIZE // len(baseline_backbones), 1),
    temperature=0.1,
)

baseline_results = evaluator.evaluate(baseline_candidates[:BATCH_SIZE], n_seeds=N_SEEDS)
baseline_hit_rate = evaluator.hit_rate(baseline_results)
baseline_mean_iptm = baseline_results['iptm'].mean()

print(f'Baseline hit rate: {baseline_hit_rate:.1%}')
print(f'Baseline mean iPTM: {baseline_mean_iptm:.3f}')

## 5.2 REINFORCE RL Sampler

In [ ]:
print('=== REINFORCE ===')

reinforce_sampler = ReinforceRLSampler(
    generator=generator,
    evaluator=evaluator,
    init_temperature=0.1,
    learning_rate=0.05,
    baseline='running_mean',
)

reinforce_history = reinforce_sampler.train(
    n_iterations=15,
    batch_size=BATCH_SIZE,
    n_seeds=N_SEEDS,
    backbone_pdbs=baseline_backbones,  # Reuse backbones for speed
)

In [ ]:
fig = reinforce_sampler.plot_learning_curve(reinforce_history, output_dir=figures_dir)
plt.show()

In [ ]:
# REINFORCE summary stats
print('REINFORCE results:')
print(f'  Initial iPTM: {reinforce_history.iloc[0]["mean_iptm"]:.3f}')
print(f'  Final iPTM:   {reinforce_history.iloc[-1]["mean_iptm"]:.3f}')
print(f'  Best iPTM:    {reinforce_history["mean_iptm"].max():.3f}')
print(f'  Initial hit rate: {reinforce_history.iloc[0]["hit_rate"]:.1%}')
print(f'  Final hit rate:   {reinforce_history.iloc[-1]["hit_rate"]:.1%}')
print(f'  Best hit rate:    {reinforce_history["hit_rate"].max():.1%}')
print(f'  Final temperature: {reinforce_history.iloc[-1]["temperature"]:.3f}')

## 5.3 Bayesian Optimisation

In [ ]:
print('=== BAYESIAN OPTIMISATION ===')

bo = BayesianOptimiser(
    generator=generator,
    evaluator=evaluator,
    param_bounds={
        'temperature': [0.05, 1.5],
        'diffusion_steps': [25, 100],
        'noise_scale': [0.5, 2.0],
    },
    n_initial=5,
)

best_params, bo_history = bo.optimise(
    n_iterations=15,
    batch_size=BATCH_SIZE,
    n_seeds=N_SEEDS,
)

print(f'\nBest hyperparameters found: {best_params}')

In [ ]:
fig = bo.plot_optimisation_trajectory(bo_history, output_dir=figures_dir)
plt.show()

In [ ]:
fig = bo.before_after_comparison(bo_history, output_dir=figures_dir)
plt.show()

## 5.4 Before vs. After Comparison

In [ ]:
# Final evaluation with best BO parameters
print('=== OPTIMISED EVALUATION ===')

optimised_backbones = generator.run_rfdiffusion(
    num_designs=max(BATCH_SIZE // 4, 4),
    diffusion_steps=int(best_params.get('diffusion_steps', 50)),
    noise_scale=best_params.get('noise_scale', 1.0),
    binder_length=70,
)

optimised_candidates = generator.run_proteinmpnn(
    backbone_pdbs=optimised_backbones,
    seqs_per_backbone=max(BATCH_SIZE // len(optimised_backbones), 1),
    temperature=best_params.get('temperature', 0.1),
)

optimised_results = evaluator.evaluate(optimised_candidates[:BATCH_SIZE], n_seeds=N_SEEDS)
optimised_hit_rate = evaluator.hit_rate(optimised_results)
optimised_mean_iptm = optimised_results['iptm'].mean()

print(f'\n--- Summary ---')
print(f'Baseline:  hit_rate={baseline_hit_rate:.1%}  iPTM={baseline_mean_iptm:.3f}')
print(f'Optimised: hit_rate={optimised_hit_rate:.1%}  iPTM={optimised_mean_iptm:.3f}')
print(f'Δ hit_rate = {optimised_hit_rate - baseline_hit_rate:+.1%}')
print(f'Δ iPTM     = {optimised_mean_iptm - baseline_mean_iptm:+.3f}')

In [ ]:
# Side-by-side iPTM distribution: baseline vs optimised
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, (label, df) in zip(axes, [
    ('Baseline (T=0.1)', baseline_results),
    (f'Optimised (T={best_params.get("temperature", 0.1):.2f})', optimised_results),
]):
    passing = df[df['passes_all']]['iptm']
    failing = df[~df['passes_all']]['iptm']
    
    bins = np.linspace(0, 1, 30)
    ax.hist(failing, bins=bins, color='#E63946', alpha=0.6, density=True, label='Failing')
    ax.hist(passing, bins=bins, color='#2E86AB', alpha=0.7, density=True, label='Passing')
    ax.axvline(df['iptm'].mean(), color='black', linestyle='--', linewidth=1.5,
               label=f'Mean={df["iptm"].mean():.3f}')
    ax.set_xlabel('iPTM', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.set_title(f'{label}\nHit rate: {evaluator.hit_rate(df):.1%}', fontweight='bold')
    ax.legend(fontsize=9)

fig.suptitle('Before vs After RL Optimisation: iPTM Distribution',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(figures_dir / 'before_after_iptm_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved before/after comparison figure.')

In [ ]:
# Save all RL histories
reinforce_history.to_csv(figures_dir / 'reinforce_history.csv', index=False)
bo_history.to_csv(figures_dir / 'bayesian_optimisation_history.csv', index=False)
print('RL history CSVs saved.')

# Print final summary
print('\n=== FINAL RL SUMMARY ===')
print(f'Target: {TARGET_NAME} ({target.pdb_id})')
print(f'Baseline hit rate:             {baseline_hit_rate:.1%}')
print(f'REINFORCE best hit rate:       {reinforce_history["hit_rate"].max():.1%}')
print(f'Bayesian Opt. best hit rate:   {bo_history["hit_rate"].max():.1%}')
print(f'Final optimised hit rate:      {optimised_hit_rate:.1%}')
print(f'Improvement:                   {optimised_hit_rate - baseline_hit_rate:+.1%}')
print(f'Best BO params: {best_params}')

## Interview Talking Points

**Q: How would you optimise sampling for a new target class?**

Use the Bayesian Optimisation approach above. It learns a surrogate model (GP) of the hit-rate landscape, so it's efficient even with expensive evaluations. For a new target class, I'd:
1. Start with 5 random explorations to build initial GP data
2. Run 15-20 BO iterations optimising temperature + diffusion steps
3. Use EI acquisition to balance exploration vs exploitation

**Q: How do you close the loop between in silico and wet lab?**

The RL loop is designed exactly for this. When wet lab data comes in:
1. Replace the `mock` backend with real iPTM from Chai-1
2. Add wet lab binary outcomes (binder/non-binder) as an additional reward signal
3. Learn a calibration mapping: iPTM → wet lab success probability
4. Use the calibrated reward in the RL update

**Q: What would you add for production use?**

1. Multi-objective reward: balance iPTM, diversity, and manufacturability
2. Constrained optimisation: exclude sequences with known immunogenic motifs
3. Active learning: prioritise candidates that maximally reduce uncertainty in GP
4. Population-based RL (evolutionary strategies) for escaping local optima